In [2]:
import os
import pandas as pd
import xarray as xr
import numpy as np

import fates_calibration_library.param_gen.param_spec as ps_module
import fates_calibration_library.param_gen.scaler as scaler_mod

import importlib

In [6]:
param_dir = '/glade/work/afoster/FATES_calibration/parameter_files'
param_data_file = 'param_list_sci.1.81.1_api.38.0.0_nwt_jan2026.xls'

default_ds = xr.open_dataset(os.path.join(param_dir, 'fates_params_api40_nwt_update_agb.nc'))

# build a lookup of pft sheets, keyed by parameter_name
xl = pd.ExcelFile(os.path.join(param_dir, param_data_file), engine="xlrd")
main = pd.read_excel(xl, sheet_name="main")
pft_sheets = {}
for sheet in xl.sheet_names:
    if sheet != "main":
        pft_sheets[f"fates_{sheet}"] = pd.read_excel(xl, sheet_name=sheet)

In [7]:
importlib.reload(ps_module)

DefaultScaler = scaler_mod.DefaultScaler
ParamSpec = ps_module.ParamSpec

scaler = DefaultScaler()

In [8]:
# construct the param_specs
specs = [
    ParamSpec.from_row(row, pft_sheet=pft_sheets.get(row["parameter_name"]))
    for _, row in main.iterrows()
]

ValueError: Parameter 'fates_leafn_vert_scaler' has param_type '<fates_calibration_library.param_gen.param_type.MultiParamType object at 0x145cc75758d0>' but root_params is empty.

In [ ]:
n_samples = 100
from scipy.stats import qmc

In [ ]:
# make a ParamEnsemble class that hold the list of ParamSpecs, it does all below work
# this validate the list to make sure all root params are generated first
# ParamEnsemble can have passes (roots first, then non-roots)
# make all methods in ParamEnsemble - actual case statement?

In [ ]:
lh = qmc.LatinHypercube(d=len(specs)).random(n=n_samples)

for i_sample, sample in enumerate(lh):
    for j, spec in enumerate(specs):
        default_val = spec.get_default_value(default_ds)
        if spec.strategy == 'default':
            value = scaler.scale(spec, lh_value=sample[j], default_value=default_val)
        elif spec.strategy == 'posterior':
            # will figure this out later
            value = draw_from_posterior